# OffsideX — Phase 1: Semi-Automated VAR Feature Extraction
## `01_feature_extraction_var.ipynb`

Extracts geometrically projected VAR features (`(N, 9)`) from 97 broadcast frames stored in `/content/drive/MyDrive/Offside_Detector/dataset` and saves outputs under `/content/drive/MyDrive/Offside_Detector/v2_var/`:
- `var_calibration.csv`: 4-click calibration (attack direction, kit RGB medians, camera slope)
- `features_var.npz`: Clean projected feature matrix
- `features_var_debug.csv`: Per-image diagnostic table


In [ ]:
!pip install -q ultralytics

import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Offside_Detector')
DATASET_DIR = BASE_DIR / 'dataset'

OFFSIDE_DIR = DATASET_DIR / 'offside'
OFFSIDE_LEFT_DIR = OFFSIDE_DIR / 'left'
ONSIDE_DIR = DATASET_DIR / 'onside'
ONSIDE_LEFT_DIR = ONSIDE_DIR / 'left'

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def count_images_in_folder(folder_path, recursive=False):
    if not folder_path.exists():
        return 0
    files = folder_path.rglob('*') if recursive else folder_path.glob('*')
    return sum(1 for f in files if f.is_file() and f.suffix.lower() in VALID_EXTS)


if not DATASET_DIR.exists():
    print(f"Dataset directory not found: {DATASET_DIR}")
else:
    n_offside_right = count_images_in_folder(OFFSIDE_DIR)
    n_offside_left = count_images_in_folder(OFFSIDE_LEFT_DIR)
    n_offside_total = n_offside_right + n_offside_left

    n_onside_right = count_images_in_folder(ONSIDE_DIR)
    n_onside_left = count_images_in_folder(ONSIDE_LEFT_DIR)
    n_onside_total = n_onside_right + n_onside_left

    total_images = n_offside_total + n_onside_total
    print(f"Offside -> right: {n_offside_right}, left: {n_offside_left}, total: {n_offside_total}")
    print(f"Onside  -> right: {n_onside_right}, left: {n_onside_left}, total: {n_onside_total}")
    print(f"Total dataset images: {total_images}")

V2_DIR = BASE_DIR / 'v2_var'
V2_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {V2_DIR}")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def load_and_align_image(image_path, is_left=False):
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Failed to read image: {image_path}")
    was_flipped = False
    if is_left:
        img = cv2.flip(img, 1)  # Normalize attacks left-to-right
        was_flipped = True
    return img, was_flipped


def get_pitch_mask(img):
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_green = np.array([30, 40, 40], dtype=np.uint8)
    upper_green = np.array([85, 255, 255], dtype=np.uint8)
    raw_mask = cv2.inRange(hsv, lower_green, upper_green)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    closed_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(closed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    pitch_mask = np.zeros((h, w), dtype=np.uint8)
    pitch_hull = None
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        pitch_hull = cv2.convexHull(largest_contour)
        cv2.drawContours(pitch_mask, [pitch_hull], -1, 255, thickness=cv2.FILLED)
    else:
        pitch_mask[int(h * 0.2):, :] = 255

    # Trim top ad boards without eroding sides
    erode_kernel = np.ones((45, 15), dtype=np.uint8)
    pitch_mask = cv2.erode(pitch_mask, erode_kernel, iterations=1)
    return pitch_mask, pitch_hull


sample_folder = OFFSIDE_LEFT_DIR if OFFSIDE_LEFT_DIR.exists() and count_images_in_folder(OFFSIDE_LEFT_DIR) > 0 else OFFSIDE_DIR
sample_images = [f for f in sample_folder.glob('*') if f.suffix.lower() in VALID_EXTS]

if sample_images:
    sample_path = sample_images[0]
    is_left_sample = 'left' in str(sample_path).lower()
    aligned_img, was_flipped = load_and_align_image(sample_path, is_left=is_left_sample)
    mask, hull = get_pitch_mask(aligned_img)
    pitch_only = cv2.bitwise_and(aligned_img, aligned_img, mask=mask)
    print(f"Shape: {aligned_img.shape} | Flipped: {was_flipped}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(aligned_img, cv2.COLOR_BGR2RGB)); axes[0].set_title(sample_path.name); axes[0].axis('off')
    axes[1].imshow(mask, cmap='gray'); axes[1].set_title("Pitch Mask (Erode 45x15)"); axes[1].axis('off')
    axes[2].imshow(cv2.cvtColor(pitch_only, cv2.COLOR_BGR2RGB)); axes[2].set_title("Pitch Isolated"); axes[2].axis('off')
    plt.tight_layout(); plt.show()


In [ ]:
from ultralytics import YOLO

det_model = YOLO('yolov8x.pt')
pose_model = YOLO('yolov8n-pose.pt')

LEGAL_KEYPOINT_INDICES = [0, 1, 2, 3, 4, 5, 6, 11, 12, 13, 14, 15, 16]  # Exclude arms per FIFA Law 11
EXCLUDED_ARM_INDICES = [7, 8, 9, 10]
MIN_CROP_HEIGHT = 120


def detect_players_and_ball(img, pitch_mask, det_model, pose_model, imgsz=1280):
    h, w = img.shape[:2]
    det_results = det_model(img, classes=[0, 32], conf=0.10, imgsz=imgsz, verbose=False)[0]
    person_boxes, ball_pos = [], None

    if det_results.boxes is not None and len(det_results.boxes) > 0:
        boxes_data = det_results.boxes.xyxy.cpu().numpy()
        classes_data = det_results.boxes.cls.cpu().numpy().astype(int)
        confs_data = det_results.boxes.conf.cpu().numpy()

        ball_candidates = []
        for i, (b, cls_id) in enumerate(zip(boxes_data, classes_data)):
            if cls_id == 0:
                person_boxes.append(b)
            elif cls_id == 32:
                ball_conf = confs_data[i]
                if ball_conf < 0.28:  # Reject white socks false positives
                    continue
                bx, by = int((b[0] + b[2]) / 2.0), int((b[1] + b[3]) / 2.0)
                bw, bh = b[2] - b[0], b[3] - b[1]
                if not (0 <= bx < w and 0 <= by < h and pitch_mask[by, bx] > 0):
                    continue
                aspect_ratio = bw / max(bh, 1e-5)
                if aspect_ratio < 0.6 or aspect_ratio > 1.6 or max(bw, bh) > 45:
                    continue
                ball_candidates.append((ball_conf, np.array([float(bx), float(by)], dtype=np.float32)))

        if ball_candidates:
            ball_candidates.sort(key=lambda x: x[0], reverse=True)
            ball_pos = ball_candidates[0][1]

    on_pitch_boxes = []
    for box in person_boxes:
        center_x = int(np.clip((box[0] + box[2]) / 2.0, 0, w - 1))
        foot_y = int(np.clip(box[3] - 2, 0, h - 1))
        if pitch_mask[foot_y, center_x] > 0:  # Foot-only test rejects sidelines
            on_pitch_boxes.append(box)

    players, pose_success, pose_fallback = [], 0, 0
    for box in on_pitch_boxes:
        x1, y1, x2, y2 = box.astype(int)
        bw, bh = x2 - x1, y2 - y1
        cx1, cy1 = max(0, x1 - int(0.10 * bw)), max(0, y1 - int(0.10 * bh))
        cx2, cy2 = min(w, x2 + int(0.10 * bw)), min(h, y2 + int(0.10 * bh))
        crop = img[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            continue

        pose_conf = 0.15
        if crop.shape[0] < MIN_CROP_HEIGHT:  # Upscale far players for Pose
            scale = 3 if crop.shape[0] < 60 else 2
            crop = cv2.resize(crop, (crop.shape[1] * scale, crop.shape[0] * scale), interpolation=cv2.INTER_LINEAR)
            pose_conf = 0.10

        pose_res = pose_model(crop, conf=pose_conf, verbose=False)[0]
        got_keypoints = (
            pose_res.keypoints is not None
            and len(pose_res.keypoints) > 0
            and pose_res.keypoints.conf is not None
        )

        if got_keypoints:
            confs_all = pose_res.keypoints.conf.cpu().numpy()
            best_k = int(np.argmax(confs_all.mean(axis=1)))
            sx, sy = (cx2 - cx1) / crop.shape[1], (cy2 - cy1) / crop.shape[0]
            conf = confs_all[best_k]
            global_xy = pose_res.keypoints.xy.cpu().numpy()[best_k] * np.array([sx, sy]) + np.array([cx1, cy1])

            legal_pts = [global_xy[idx] for idx in LEGAL_KEYPOINT_INDICES if conf[idx] > 0.25]
            if legal_pts:
                max_point = max(legal_pts, key=lambda pt: pt[0])
                max_x = float(max_point[0])
            else:
                max_x = float(x2)
                max_point = np.array([float(x2), (y1 + y2) / 2.0])

            players.append({
                'box': box, 'keypoints_xy': global_xy, 'keypoints_conf': conf,
                'max_x': max_x, 'max_point': max_point, 'box_height': float(bh),
                'box_width': float(bw), 'center_x': (x1 + x2) / 2.0,
                'center_y': (y1 + y2) / 2.0, 'pose_source': 'keypoints',
            })
            pose_success += 1
        else:
            max_x = float(box[2]) - 0.10 * float(bw)
            players.append({
                'box': box, 'keypoints_xy': None, 'keypoints_conf': None,
                'max_x': max_x, 'max_point': np.array([max_x, (y1 + y2) / 2.0]),
                'box_height': float(bh), 'box_width': float(bw),
                'center_x': (x1 + x2) / 2.0, 'center_y': (y1 + y2) / 2.0,
                'pose_source': 'bbox_fallback',
            })
            pose_fallback += 1

    print(f"Det: {len(person_boxes)} | OnPitch: {len(on_pitch_boxes)} | Pose: {pose_success} | Fallback: {pose_fallback} | Ball: {ball_pos is not None}")
    return players, ball_pos


if 'aligned_img' in locals() and 'mask' in locals():
    detected_players, detected_ball = detect_players_and_ball(aligned_img, mask, det_model, pose_model)
    vis_img = aligned_img.copy()
    for p in detected_players:
        b = p['box'].astype(int)
        box_color = (0, 200, 255) if p['pose_source'] == 'bbox_fallback' else (255, 200, 0)
        cv2.rectangle(vis_img, (b[0], b[1]), (b[2], b[3]), box_color, 2)
        if p['keypoints_xy'] is not None:
            for idx in LEGAL_KEYPOINT_INDICES:
                if p['keypoints_conf'][idx] > 0.25:
                    cv2.circle(vis_img, tuple(p['keypoints_xy'][idx].astype(int)), 4, (0, 255, 0), -1)
            for idx in EXCLUDED_ARM_INDICES:
                if p['keypoints_conf'][idx] > 0.25:
                    cv2.circle(vis_img, tuple(p['keypoints_xy'][idx].astype(int)), 4, (0, 0, 255), -1)
        cv2.circle(vis_img, tuple(p['max_point'].astype(int)), 7, (0, 255, 255), -1)

    if detected_ball is not None:
        cv2.circle(vis_img, (int(detected_ball[0]), int(detected_ball[1])), 8, (255, 0, 255), -1)
        cv2.putText(vis_img, "Ball", (int(detected_ball[0]) - 15, int(detected_ball[1]) - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2)

    plt.figure(figsize=(15, 9)); plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Top-Down Detection | Total: {len(detected_players)} | Ball: {'Yes' if detected_ball is not None else 'No'}")
    plt.axis('off'); plt.show()


In [ ]:
GRASS_GREEN_LOWER = np.array([36, 40, 40], dtype=np.uint8)  # Preserve yellow kits
GRASS_GREEN_UPPER = np.array([85, 255, 255], dtype=np.uint8)
GRASS_RATIO_THRESHOLD = 0.70

COLOR_WEIGHTS = np.array([0.50, 1.40, 1.40], dtype=np.float32)  # De-emphasize luminance L
MAX_TEAM_COLOR_DIST = 44.0  # Strict cutoff keeps GKs out of teams
MAX_AMBIGUITY_RATIO = 0.80  # Reject ambiguous kits midway between teams

GK_MIN_X_RATIO = 0.65
GK_MIN_Y_RATIO = 0.08
GK_MAX_Y_RATIO = 0.92
GK_MIN_ATT_DIST = 35.0


def rgb_to_weighted_lab(rgb):
    bgr = np.uint8([[[int(rgb[2]), int(rgb[1]), int(rgb[0])]]])
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2Lab)[0, 0].astype(np.float32)
    return lab * COLOR_WEIGHTS


def get_jersey_lab_color(img, player):
    h_img, w_img = img.shape[:2]
    x1, y1, x2, y2 = player['box'].astype(int)
    bh, bw = y2 - y1, x2 - x1

    torso_crop_done = False
    if player['keypoints_xy'] is not None and player['keypoints_conf'] is not None:
        kp, kc = player['keypoints_xy'], player['keypoints_conf']
        valid_pts = [kp[idx] for idx in [5, 6, 11, 12] if kc[idx] > 0.25]
        if len(valid_pts) >= 2:
            cx = float(np.mean([pt[0] for pt in valid_pts]))
            cy = float(np.mean([pt[1] for pt in valid_pts]))
            half_size = int(np.clip(0.11 * bh, 6, 30))
            crop_x1, crop_x2 = max(0, int(cx - half_size)), min(w_img, int(cx + half_size))
            crop_y1, crop_y2 = max(0, int(cy - half_size)), min(h_img, int(cy + half_size))
            if crop_x2 > crop_x1 and crop_y2 > crop_y1:
                torso_crop_done = True

    if not torso_crop_done:
        crop_y1, crop_y2 = max(0, y1 + int(0.20 * bh)), min(h_img, y1 + int(0.46 * bh))
        crop_x1, crop_x2 = max(0, x1 + int(0.28 * bw)), min(w_img, x1 + int(0.72 * bw))
        if crop_y2 <= crop_y1 or crop_x2 <= crop_x1:
            crop_y1, crop_y2 = max(0, y1 + int(0.20 * bh)), min(h_img, y1 + int(0.45 * bh))
            crop_x1, crop_x2 = max(0, x1 + int(0.15 * bw)), min(w_img, x2 - int(0.15 * bw))

    jersey_crop = img[crop_y1:crop_y2, crop_x1:crop_x2]
    if jersey_crop.size == 0:
        return np.array([50.0, 0.0, 0.0], dtype=np.float32)

    hsv_crop = cv2.cvtColor(jersey_crop, cv2.COLOR_BGR2HSV)
    green_mask = cv2.inRange(hsv_crop, GRASS_GREEN_LOWER, GRASS_GREEN_UPPER)
    total_px = max(jersey_crop.shape[0] * jersey_crop.shape[1], 1)
    green_ratio = green_mask.sum() / 255.0 / total_px

    not_green = None
    if green_ratio < GRASS_RATIO_THRESHOLD:
        mask_ng = cv2.bitwise_not(green_mask)
        if mask_ng.sum() / 255 >= 4:
            not_green = mask_ng

    lab_crop = cv2.cvtColor(jersey_crop, cv2.COLOR_BGR2Lab)
    if not_green is not None and (not_green > 0).sum() > 0:
        return np.median(lab_crop[not_green > 0], axis=0).astype(np.float32)
    return np.median(lab_crop.reshape(-1, 3), axis=0).astype(np.float32)


def classify_teams_calibrated(players, img, att_rgb, def_rgb):
    h_img, w_img = img.shape[:2]
    if not players:
        return [], [], None, {'excluded': []}

    att_lab = rgb_to_weighted_lab(att_rgb)
    def_lab = rgb_to_weighted_lab(def_rgb)
    attackers, defenders, excluded = [], [], []

    for p in players:
        c_lab = get_jersey_lab_color(img, p) * COLOR_WEIGHTS
        d_att = float(np.linalg.norm(c_lab - att_lab))
        d_def = float(np.linalg.norm(c_lab - def_lab))
        min_d, max_d = min(d_att, d_def), max(d_att, d_def)

        p['team_dist'], p['d_att'], p['d_def'] = min_d, d_att, d_def
        if (min_d < MAX_TEAM_COLOR_DIST) and (min_d / (max_d + 1e-6) < MAX_AMBIGUITY_RATIO):
            (attackers if d_att <= d_def else defenders).append(p)
        else:
            excluded.append(p)

    # Never drop frames under heavy shadow
    if not attackers and excluded:
        cands_att = sorted([p for p in excluded if p['d_att'] <= p['d_def'] and p['d_att'] < 85.0], key=lambda p: p['d_att'])
        if not cands_att:
            cands_att = [min(excluded, key=lambda p: p['d_att'])]
        attackers.extend(cands_att)
        excluded = [x for x in excluded if not any(x is p for p in cands_att)]

    if not defenders and excluded:
        cands_def = sorted([p for p in excluded if p['d_def'] < p['d_att'] and p['d_def'] < 85.0], key=lambda p: p['d_def'])
        if not cands_def:
            cands_def = [min(excluded, key=lambda p: p['d_def'])]
        defenders.extend(cands_def)
        excluded = [x for x in excluded if not any(x is p for p in cands_def)]

    gk_candidates = [
        (p['max_x'], idx) for idx, p in enumerate(excluded)
        if p['max_x'] / w_img >= GK_MIN_X_RATIO
        and GK_MIN_Y_RATIO <= p['center_y'] / h_img <= GK_MAX_Y_RATIO
        and p['d_att'] >= GK_MIN_ATT_DIST
    ]
    goalkeeper = excluded.pop(max(gk_candidates)[1]) if gk_candidates else None

    # Catch GK who slipped into attackers near goal
    if goalkeeper is None and len(attackers) >= 2 and defenders:
        max_def_x = max(p['max_x'] for p in defenders)
        right_att = max(attackers, key=lambda p: p['max_x'])
        if right_att['max_x'] > max_def_x + 0.08 * w_img and right_att['max_x'] >= 0.62 * w_img and right_att['d_att'] > 26.0:
            attackers = [p for p in attackers if p is not right_att]  # Identity guard avoids NumPy Eq
            goalkeeper = right_att

    return attackers, defenders, goalkeeper, {'excluded': excluded}


print("Loaded Color-Anchored LAB & GK guard helpers.")


In [ ]:
def project_x_along_slope(player, ref_y, pitch_slope):
    foot_y = float(player['box'][3])
    return float(player['max_x'] + pitch_slope * (ref_y - foot_y))


def get_var_defensive_line(defenders, goalkeeper, pitch_slope):
    opp = list(defenders) + ([goalkeeper] if goalkeeper is not None else [])
    ref_y = float(np.mean([p['box'][3] for p in opp]))
    opp_sorted = sorted(opp, key=lambda p: project_x_along_slope(p, ref_y, pitch_slope), reverse=True)
    if goalkeeper is not None and len(opp_sorted) >= 2:
        return opp_sorted[1], opp_sorted
    return opp_sorted[0], opp_sorted


def draw_var_line(vis, x0, y0, slope, color, thickness=2, label=None, label_y=30):
    h, w = vis.shape[:2]
    x_top = int(np.clip(x0 + slope * (0 - y0), -w, 2 * w))
    x_bot = int(np.clip(x0 + slope * (h - y0), -w, 2 * w))
    cv2.line(vis, (x_top, 0), (x_bot, h), color, thickness)
    if label:
        lx = int(np.clip(x0 + slope * (label_y - y0) + 5, 5, w - 70))
        cv2.putText(vis, label, (lx, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)


def run_var_pipeline(image_path, cal, det_model, pose_model, label='', draw=True):
    img, _ = load_and_align_image(image_path, is_left=bool(cal['final_flip']))
    pitch_mask, _ = get_pitch_mask(img)
    players, ball_pos = detect_players_and_ball(img, pitch_mask, det_model, pose_model)

    if len(players) < 2:
        return None, None, f'Too few players ({len(players)})'

    attackers, defenders, goalkeeper, extra = classify_teams_calibrated(
        players, img, cal['att_rgb'], cal['def_rgb']
    )
    if not attackers or not defenders:
        return None, None, f'Empty team (ATT={len(attackers)}, DEF={len(defenders)})'

    h, w = img.shape[:2]
    pitch_slope = float(cal['pitch_slope'])

    d1, opp_sorted = get_var_defensive_line(defenders, goalkeeper, pitch_slope)
    k = 1 if (goalkeeper is not None and len(opp_sorted) >= 2) else 0
    d2 = opp_sorted[k + 1] if len(opp_sorted) > k + 1 else d1
    d1_foot_y = float(d1['box'][3])

    att_sorted = sorted(attackers, key=lambda p: project_x_along_slope(p, d1_foot_y, pitch_slope), reverse=True)
    a1 = att_sorted[0]
    a2 = att_sorted[1] if len(att_sorted) > 1 else a1

    a1_proj_x = project_x_along_slope(a1, d1_foot_y, pitch_slope)
    a2_proj_x = project_x_along_slope(a2, d1_foot_y, pitch_slope)
    d2_proj_x = project_x_along_slope(d2, d1_foot_y, pitch_slope)

    var_diff_x = (a1_proj_x - float(d1['max_x'])) / w
    raw_diff_x = (float(a1['max_x']) - float(d1['max_x'])) / w

    feats = np.array([
        var_diff_x,
        raw_diff_x,
        pitch_slope,
        (a2_proj_x - float(d1['max_x'])) / w,
        (float(a1['box'][3]) - d1_foot_y) / h,
        a1['box_height'] / (d1['box_height'] + 1e-6),
        (float(d1['max_x']) - d2_proj_x) / w,
        float(a1['max_x']) / w,
        ((float(a1['max_x']) - ball_pos[0]) / w) if ball_pos is not None else 0.0,
    ], dtype=np.float32)

    info = {
        'n_att': len(attackers), 'n_def': len(defenders),
        'gk': goalkeeper is not None, 'ball': ball_pos is not None,
        'var_diff_x': var_diff_x, 'raw_diff_x': raw_diff_x,
        'pitch_slope': pitch_slope, 'predicted_offside': bool(var_diff_x > 0.0),
    }

    vis = None
    if draw:
        vis = img.copy()
        for p in attackers:
            b = p['box'].astype(int)
            cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (0, 0, 255), 2)
            cv2.circle(vis, tuple(p['max_point'].astype(int)), 5, (0, 0, 255), -1)
        for p in defenders:
            b = p['box'].astype(int)
            cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (255, 0, 0), 2)
            cv2.circle(vis, tuple(p['max_point'].astype(int)), 5, (255, 0, 0), -1)
        for p in extra.get('excluded', []):
            b = p['box'].astype(int)
            cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (150, 150, 150), 2)
            cv2.putText(vis, f"{p['team_dist']:.0f}", (b[0], b[1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 2)
        if goalkeeper:
            b = goalkeeper['box'].astype(int)
            cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 3)
            cv2.putText(vis, 'GK', (b[0], b[1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)

        draw_var_line(vis, float(a1['max_x']), float(a1['box'][3]), pitch_slope, (0, 0, 255), 2, 'ATT', 28)
        draw_var_line(vis, float(d1['max_x']), d1_foot_y, pitch_slope, (255, 0, 0), 2, 'DEF', 54)

        if ball_pos is not None:
            cv2.circle(vis, (int(ball_pos[0]), int(ball_pos[1])), 8, (255, 0, 255), -1)
            cv2.putText(vis, 'Ball', (int(ball_pos[0]) - 15, int(ball_pos[1]) - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 255), 2)

        pred_txt = 'OFFSIDE' if var_diff_x > 0 else 'ONSIDE'
        pred_col = (0, 0, 255) if var_diff_x > 0 else (0, 200, 0)
        cv2.putText(vis, f"{label} -> VAR: {pred_txt} (vdiff={var_diff_x:+.3f}, slope={pitch_slope:+.2f})",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.65, pred_col, 2)

    return vis, (feats, info), None


print("Loaded VAR perspective projection helpers.")


In [ ]:
import csv, base64, json
from google.colab import output as colab_output
from IPython.display import display, HTML, JSON

CAL_CSV = V2_DIR / 'var_calibration.csv'
CAL_FIELDS = ['image_id', 'final_flip', 'att_r', 'att_g', 'att_b', 'def_r', 'def_g', 'def_b', 'pitch_slope']


def list_dataset():
    items = []
    for folder, is_left, y in [(OFFSIDE_DIR, False, 1), (OFFSIDE_LEFT_DIR, True, 1),
                               (ONSIDE_DIR, False, 0), (ONSIDE_LEFT_DIR, True, 0)]:
        if not folder.exists():
            continue
        for f in sorted(folder.iterdir()):
            if f.is_file() and f.suffix.lower() in VALID_EXTS:
                items.append({'id': str(f.relative_to(DATASET_DIR)), 'path': f, 'is_left': is_left, 'y': y})
    return items


def load_calibrations(path):
    cals = {}
    if path.exists():
        with open(path, encoding='utf-8') as f:
            for row in csv.DictReader(f):
                cals[row['image_id']] = {
                    'final_flip': row['final_flip'] == 'True',
                    'att_rgb': (int(row['att_r']), int(row['att_g']), int(row['att_b'])),
                    'def_rgb': (int(row['def_r']), int(row['def_g']), int(row['def_b'])),
                    'pitch_slope': float(row['pitch_slope']),
                }
    return cals


def save_calibrations(path, cals):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=CAL_FIELDS)
        w.writeheader()
        for img_id, c in cals.items():
            w.writerow({
                'image_id': img_id,
                'final_flip': str(bool(c['final_flip'])),
                'att_r': c['att_rgb'][0], 'att_g': c['att_rgb'][1], 'att_b': c['att_rgb'][2],
                'def_r': c['def_rgb'][0], 'def_g': c['def_rgb'][1], 'def_b': c['def_rgb'][2],
                'pitch_slope': f"{float(c['pitch_slope']):.5f}",
            })


def sample_median_rgb(img_bgr, nx, ny, patch_half=3):
    h, w = img_bgr.shape[:2]
    px = int(np.clip(round(nx * w), 0, w - 1))
    py = int(np.clip(round(ny * h), 0, h - 1))
    patch = img_bgr[max(0, py - patch_half):min(h, py + patch_half + 1),
                    max(0, px - patch_half):min(w, px + patch_half + 1)]
    med_bgr = np.median(patch.reshape(-1, 3), axis=0)
    return (int(med_bgr[2]), int(med_bgr[1]), int(med_bgr[0]))


def encode_preview_b64(img_bgr, max_w=900):
    h, w = img_bgr.shape[:2]
    if w > max_w:
        img_bgr = cv2.resize(img_bgr, (max_w, int(h * max_w / w)), interpolation=cv2.INTER_AREA)
    _, buf = cv2.imencode('.jpg', img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.tobytes()).decode('ascii')


_all_items = list_dataset()
_by_id = {it['id']: it for it in _all_items}
_cals = load_calibrations(CAL_CSV)
_todo_ids = [it['id'] for it in _all_items if it['id'] not in _cals]
_state = {'pos': 0, 'history': []}


def _make_payload():
    total, done = len(_all_items), len(_cals)
    if _state['pos'] >= len(_todo_ids):
        return {'finished': True, 'done': done, 'total': total}
    img_id = _todo_ids[_state['pos']]
    it = _by_id[img_id]
    img, _ = load_and_align_image(it['path'], is_left=it['is_left'])
    return {
        'finished': False, 'image_id': img_id,
        'y_text': 'OFFSIDE' if it['y'] == 1 else 'ONSIDE',
        'is_left_default': bool(it['is_left']),
        'pos': _state['pos'] + 1, 'remaining': len(_todo_ids),
        'done': done, 'total': total, 'b64': encode_preview_b64(img),
    }


def _cb_save(payload):
    img_id = payload['image_id']
    it = _by_id[img_id]
    final_flip = (bool(it['is_left']) != bool(payload['user_flip']))
    img, _ = load_and_align_image(it['path'], is_left=final_flip)
    h, w = img.shape[:2]

    att_rgb = sample_median_rgb(img, payload['att_nx'], payload['att_ny'])
    def_rgb = sample_median_rgb(img, payload['def_nx'], payload['def_ny'])

    if payload.get('zero_slope', False) or payload.get('l1_nx') is None:
        slope = 0.0
    else:
        dx = (payload['l2_nx'] - payload['l1_nx']) * w
        dy = (payload['l2_ny'] - payload['l1_ny']) * h
        slope = float(np.clip(dx / dy, -2.5, 2.5)) if abs(dy) > 5.0 else 0.0

    _cals[img_id] = {'final_flip': final_flip, 'att_rgb': att_rgb, 'def_rgb': def_rgb, 'pitch_slope': slope}
    save_calibrations(CAL_CSV, _cals)
    _state['history'].append(_state['pos'])
    _state['pos'] += 1
    return JSON(_make_payload())


def _cb_skip():
    if _state['pos'] < len(_todo_ids):
        _state['pos'] += 1
    return JSON(_make_payload())


def _cb_undo():
    if _state['history']:
        prev_pos = _state['history'].pop()
        _cals.pop(_todo_ids[prev_pos], None)
        save_calibrations(CAL_CSV, _cals)
        _state['pos'] = prev_pos
    return JSON(_make_payload())


colab_output.register_callback('var_cal_save', _cb_save)
colab_output.register_callback('var_cal_skip', _cb_skip)
colab_output.register_callback('var_cal_undo', _cb_undo)

initial_json = json.dumps(_make_payload(), ensure_ascii=True)

html_ui = f"""
<div id="var-app" style="font-family:'Segoe UI',sans-serif; background:#181a20; color:#f1f3f5; padding:14px; border-radius:10px; max-width:940px; border:1px solid #333;">
  <div id="var-header" style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:8px;">
    <div id="var-progress" style="font-size:15px; font-weight:bold;"></div>
    <div id="var-filename" style="font-size:12px; color:#adb5bd;"></div>
  </div>
  <div id="var-step-bar" style="background:#252a36; padding:10px 14px; border-radius:8px; margin-bottom:10px; font-size:15px; font-weight:bold; text-align:center; border:1px solid #3b4252;"></div>
  <div style="text-align:center;">
    <canvas id="var-canvas" style="max-width:100%; border:2px solid #495057; border-radius:6px; cursor:crosshair; background:#000;"></canvas>
  </div>
  <div style="display:flex; justify-content:center; flex-wrap:wrap; gap:8px; margin-top:12px;">
    <button id="btn-save"  style="background:#2e7d32; color:#fff; border:none; padding:9px 18px; border-radius:6px; font-weight:bold; cursor:pointer; font-size:14px;">Save &amp; Next (Space)</button>
    <button id="btn-flip"  style="background:#1565c0; color:#fff; border:none; padding:9px 14px; border-radius:6px; font-weight:bold; cursor:pointer; font-size:13px;">Flip Direction (F)</button>
    <button id="btn-zero"  style="background:#6a1b9a; color:#fff; border:none; padding:9px 14px; border-radius:6px; font-weight:bold; cursor:pointer; font-size:13px;">Vertical 0 deg (Z)</button>
    <button id="btn-reset" style="background:#ef6c00; color:#fff; border:none; padding:9px 14px; border-radius:6px; font-weight:bold; cursor:pointer; font-size:13px;">Reset Clicks (R)</button>
    <button id="btn-undo"  style="background:#455a64; color:#fff; border:none; padding:9px 14px; border-radius:6px; font-weight:bold; cursor:pointer; font-size:13px;">Undo (U)</button>
    <button id="btn-skip"  style="background:#424242; color:#fff; border:none; padding:9px 14px; border-radius:6px; font-weight:bold; cursor:pointer; font-size:13px;">Skip</button>
  </div>
</div>
<script>
(() => {{
  let cur = {initial_json};
  let userFlip = false, zeroSlope = false, pts = [], imgObj = new Image(), busy = false;
  const canvas = document.getElementById('var-canvas'), ctx = canvas.getContext('2d');
  const elProg = document.getElementById('var-progress'), elFile = document.getElementById('var-filename'), elStep = document.getElementById('var-step-bar');
  const btnSave = document.getElementById('btn-save'), btnFlip = document.getElementById('btn-flip'), btnZero = document.getElementById('btn-zero');
  const btnReset = document.getElementById('btn-reset'), btnUndo = document.getElementById('btn-undo'), btnSkip = document.getElementById('btn-skip');

  function updateStepBanner() {{
    if (cur.finished) {{
      elProg.innerText = `Calibration complete: ${{cur.done}} / ${{cur.total}} saved to v2_var/var_calibration.csv`;
      elFile.innerText = '';
      elStep.innerHTML = 'All frames calibrated. Run the next cell to extract features.';
      elStep.style.color = '#69f0ae';
      canvas.style.display = 'none';
      btnSave.disabled = true;
      return;
    }}
    canvas.style.display = 'inline-block';
    const pct = Math.round((cur.done / Math.max(cur.total, 1)) * 100);
    elProg.innerText = `Frame ${{cur.pos}}/${{cur.remaining}} | Done: ${{cur.done}}/${{cur.total}} (${{pct}}%) | ${{cur.y_text}}`;
    elFile.innerText = cur.image_id;
    if (pts.length === 0) elStep.innerHTML = '1. Click an <span style="color:#ff5252;">ATTACKER</span> jersey (attack facing right)';
    else if (pts.length === 1) elStep.innerHTML = '2. Click a <span style="color:#448aff;">DEFENDER</span> jersey';
    else if (pts.length === 2 && !zeroSlope) elStep.innerHTML = '3. Click <span style="color:#ffd740;">Point 1</span> on the penalty box or turf line (or Z for vertical)';
    else if (pts.length === 3 && !zeroSlope) elStep.innerHTML = '4. Click <span style="color:#ffd740;">Point 2</span> along the same line';
    else elStep.innerHTML = 'Verify the yellow VAR guide and press <span style="color:#69f0ae;">Save &amp; Next (Space)</span>';
    btnSave.style.opacity = (pts.length >= 4 || (pts.length >= 2 && zeroSlope)) ? '1.0' : '0.45';
  }}

  function redraw() {{
    if (!imgObj.complete || !canvas.width) return;
    ctx.save(); ctx.clearRect(0, 0, canvas.width, canvas.height);
    if (userFlip) {{ ctx.translate(canvas.width, 0); ctx.scale(-1, 1); }}
    ctx.drawImage(imgObj, 0, 0, canvas.width, canvas.height); ctx.restore();
    ctx.fillStyle = 'rgba(0,0,0,0.55)'; ctx.fillRect(10, 10, 175, 28);
    ctx.fillStyle = '#fff'; ctx.font = 'bold 14px sans-serif'; ctx.fillText('Attack Direction --->', 20, 29);
    if (pts.length >= 4 && !zeroSlope) {{
      const x1 = pts[2].nx * canvas.width, y1 = pts[2].ny * canvas.height;
      const x2 = pts[3].nx * canvas.width, y2 = pts[3].ny * canvas.height, dy = y2 - y1;
      if (Math.abs(dy) > 3) {{
        const s = (x2 - x1) / dy;
        ctx.strokeStyle = '#ffd740'; ctx.lineWidth = 3; ctx.setLineDash([8, 5]);
        ctx.beginPath(); ctx.moveTo(x1 + s * (0 - y1), 0); ctx.lineTo(x1 + s * (canvas.height - y1), canvas.height); ctx.stroke(); ctx.setLineDash([]);
      }}
    }} else if (zeroSlope && pts.length >= 2) {{
      const xm = canvas.width * 0.65;
      ctx.strokeStyle = '#ce93d8'; ctx.lineWidth = 2.5; ctx.setLineDash([6, 5]);
      ctx.beginPath(); ctx.moveTo(xm, 0); ctx.lineTo(xm, canvas.height); ctx.stroke(); ctx.setLineDash([]);
    }}
    const styles = [{{color:'#ff1744',label:'ATT'}},{{color:'#2979ff',label:'DEF'}},{{color:'#ffd740',label:'L1'}},{{color:'#ffd740',label:'L2'}}];
    pts.forEach((p, i) => {{
      const x = p.nx * canvas.width, y = p.ny * canvas.height;
      ctx.beginPath(); ctx.arc(x, y, 7, 0, Math.PI * 2); ctx.fillStyle = styles[i].color; ctx.fill();
      ctx.lineWidth = 2; ctx.strokeStyle = '#fff'; ctx.stroke();
      ctx.fillStyle = '#fff'; ctx.font = 'bold 12px sans-serif'; ctx.fillText(styles[i].label, x + 10, y + 4);
    }});
  }}

  function loadCurrent(data) {{
    cur = data; userFlip = false; zeroSlope = false; pts = []; updateStepBanner();
    if (!cur.finished) {{
      imgObj = new Image();
      imgObj.onload = () => {{ canvas.width = imgObj.width; canvas.height = imgObj.height; redraw(); }};
      imgObj.src = cur.b64;
    }}
  }}

  canvas.addEventListener('click', (e) => {{
    if (cur.finished || busy || pts.length >= 4 || (zeroSlope && pts.length >= 2)) return;
    const r = canvas.getBoundingClientRect();
    pts.push({{nx:(e.clientX - r.left) / r.width, ny:(e.clientY - r.top) / r.height}});
    updateStepBanner(); redraw();
  }});

  async function doSave() {{
    if (cur.finished || busy || pts.length < 2 || (pts.length < 4 && !zeroSlope)) return;
    busy = true; elStep.innerText = 'Saving...';
    const payload = {{
      image_id: cur.image_id, user_flip: userFlip, zero_slope: zeroSlope,
      att_nx: pts[0].nx, att_ny: pts[0].ny, def_nx: pts[1].nx, def_ny: pts[1].ny,
      l1_nx: pts.length >= 4 ? pts[2].nx : null, l1_ny: pts.length >= 4 ? pts[2].ny : null,
      l2_nx: pts.length >= 4 ? pts[3].nx : null, l2_ny: pts.length >= 4 ? pts[3].ny : null,
    }};
    try {{
      const res = await google.colab.kernel.invokeFunction('var_cal_save', [payload], {{}});
      const raw = res.data['application/json'];
      busy = false; loadCurrent((typeof raw === 'string') ? JSON.parse(raw) : raw);
    }} catch (err) {{ busy = false; elStep.innerText = 'Save error: ' + err; }}
  }}

  btnSave.onclick = doSave;
  btnFlip.onclick = () => {{ userFlip = !userFlip; pts = []; zeroSlope = false; updateStepBanner(); redraw(); }};
  btnZero.onclick = () => {{ zeroSlope = true; pts = pts.slice(0, 2); updateStepBanner(); redraw(); }};
  btnReset.onclick = () => {{ pts = []; zeroSlope = false; updateStepBanner(); redraw(); }};
  btnUndo.onclick = async () => {{
    if (busy) return; busy = true;
    try {{
      const res = await google.colab.kernel.invokeFunction('var_cal_undo', [], {{}});
      const raw = res.data['application/json'];
      loadCurrent((typeof raw === 'string') ? JSON.parse(raw) : raw);
    }} finally {{ busy = false; }}
  }};
  btnSkip.onclick = async () => {{
    if (busy || cur.finished) return; busy = true;
    try {{
      const res = await google.colab.kernel.invokeFunction('var_cal_skip', [], {{}});
      const raw = res.data['application/json'];
      loadCurrent((typeof raw === 'string') ? JSON.parse(raw) : raw);
    }} finally {{ busy = false; }}
  }};
  document.addEventListener('keydown', (e) => {{
    if (cur.finished || busy) return;
    if ((e.code === 'Space' || e.code === 'Enter') && (pts.length >= 4 || (pts.length >= 2 && zeroSlope))) {{
      e.preventDefault(); doSave();
    }}
    else if (e.key.toLowerCase() === 'f') btnFlip.click();
    else if (e.key.toLowerCase() === 'r') btnReset.click();
    else if (e.key.toLowerCase() === 'z') btnZero.click();
    else if (e.key.toLowerCase() === 'u') btnUndo.click();
  }});
  loadCurrent(cur);
}})();
</script>
"""
display(HTML(html_ui))


In [ ]:
import io, contextlib
import pandas as pd
from tqdm.auto import tqdm

FEATURE_NAMES = [
    'var_diff_x',
    'raw_diff_x',
    'pitch_slope',
    'second_att_var_diff',
    'y_diff',
    'scale_ratio',
    'def_line_spread',
    'attacker_x_norm',
    'ball_x_diff',
]

items = list_dataset()
cals = load_calibrations(V2_DIR / 'var_calibration.csv')
todo = [it for it in items if it['id'] in cals]

print(f"Total dataset: {len(items)} | Calibrated: {len(todo)}")
if len(todo) == 0:
    raise RuntimeError("No calibrated images found; run the calibration cell first.")

X_list, y_list, names, rows, skipped = [], [], [], [], []
for it in tqdm(todo):
    cal = cals[it['id']]
    with contextlib.redirect_stdout(io.StringIO()):
        _, out, err = run_var_pipeline(it['path'], cal, det_model, pose_model, draw=False)
    if err is not None:
        skipped.append((it['id'], err))
        continue
    feats, info = out
    X_list.append(feats)
    y_list.append(it['y'])
    names.append(it['id'])
    rows.append({
        'image_id': it['id'], 'y': it['y'], 'final_flip': cal['final_flip'],
        **dict(zip(FEATURE_NAMES, feats.tolist())),
        'n_att': info['n_att'], 'n_def': info['n_def'], 'gk': info['gk'], 'ball': info['ball'],
        'var_pred': int(feats[0] > 0), 'raw_pred': int(feats[1] > 0),
    })

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list, dtype=np.int64)

np.savez(V2_DIR / 'features_var.npz', X=X, y=y, names=np.array(names), feature_names=np.array(FEATURE_NAMES))
dbg = pd.DataFrame(rows)
dbg.to_csv(V2_DIR / 'features_var_debug.csv', index=False, encoding='utf-8')

print(f"Saved v2_var/features_var.npz | X: {X.shape} | y: {y.shape} (offside={int(y.sum())}, onside={int((y==0).sum())})")
if skipped:
    print(f"Skipped {len(skipped)}: {skipped[:5]}")

raw_acc = float(((X[:, 1] > 0).astype(int) == y).mean())
var_pred = (X[:, 0] > 0).astype(int)
var_acc = float((var_pred == y).mean())
tp = int(((var_pred == 1) & (y == 1)).sum()); tn = int(((var_pred == 0) & (y == 0)).sum())
fp = int(((var_pred == 1) & (y == 0)).sum()); fn = int(((var_pred == 0) & (y == 1)).sum())

print(f"Raw 2D Accuracy (raw_diff_x > 0): {raw_acc * 100:.1f}%")
print(f"VAR Accuracy    (var_diff_x > 0): {var_acc * 100:.1f}%")
print(f"TP={tp} | FN={fn} | TN={tn} | FP={fp}")


In [ ]:
import io, contextlib
import pandas as pd

dbg = pd.read_csv(V2_DIR / 'features_var_debug.csv')
cals = load_calibrations(V2_DIR / 'var_calibration.csv')
by_id = {it['id']: it for it in list_dataset()}
wrong = dbg[dbg['var_pred'] != dbg['y']].reset_index(drop=True)

print(f"Errors: {len(wrong)} / {len(dbg)} (VAR Acc: {(1 - len(wrong) / max(len(dbg), 1)) * 100:.1f}%)")


def review(start=0, n=4):
    rows = wrong.iloc[start:start + n]
    if rows.empty:
        print("No remaining error frames.")
        return
    fig, axes = plt.subplots(2, 2, figsize=(22, 13))
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
        it, cal = by_id[r['image_id']], cals[r['image_id']]
        truth = 'OFFSIDE' if r['y'] == 1 else 'ONSIDE'
        with contextlib.redirect_stdout(io.StringIO()):
            vis, _, _ = run_var_pipeline(it['path'], cal, det_model, pose_model, label=truth, draw=True)
        if vis is not None:
            ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(
            f"Truth: {truth} | VAR: {'OFFSIDE' if r['var_pred'] else 'ONSIDE'}\n"
            f"vdiff={r['var_diff_x']:+.3f} (raw={r['raw_diff_x']:+.3f}) | slope={r['pitch_slope']:+.2f} | {r['image_id'][:45]}",
            fontsize=10,
        )
    plt.tight_layout(); plt.show()


review(0)
